# The Schrodinger equation: the systems with exact answers

Textbook problems solved with the same Numerov integrator used in the
scattering problem, now looking for eigenvalues: the values of $E$ for which
$u$ vanishes at both ends.

$$u''(x) = 2\,[V(x) - E]\,u(x), \qquad u(x_0) = u(x_1) = 0$$

The method lives in `schrodinger.py`, next to this notebook, and rests on the
**oscillation theorem**: the number of nodes of $u$ counts how many
eigenvalues lie below $E$. A bisection on the node count corners each $E_n$,
and `brentq` refines it.

Every system here has a closed form, which is the whole point: each one is an
opportunity to catch the integrator being wrong.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import schrodinger as sq   # the file next to this notebook
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('ready')

---
## 1. Harmonic oscillator

$$V(x) = \tfrac{1}{2}x^2 \qquad\Rightarrow\qquad E_n = n + \tfrac{1}{2} \quad (\hbar = m = \omega = 1)$$

Here $n$ is the number of nodes of the wavefunction, and the $\tfrac12$ is the
zero-point energy: even in the ground state the particle is not at rest, by
Heisenberg. Equally spaced levels are the signature of the harmonic oscillator.

Near its bottom, an optical trap for cold atoms is exactly this potential.
Squeezing from three dimensions to two means making one frequency much larger
than the others, freezing that degree of freedom into its zero-point state.

In [ ]:
# how many states to draw
n_show = 4

levels = sq.eigenvalues(sq.V_oscillator, -8, 8, 0.0, n_show + 1.0,
                        n_states=n_show, dx=4e-3)
xx = np.linspace(-5, 5, 400)
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.plot(xx, sq.V_oscillator(xx), 'k-', lw=2, label='V(x) = x²/2')
for E, n in levels:
    x, u = sq.eigenfunction(sq.V_oscillator, E, -8, 8, dx=4e-3)
    m = np.abs(x) <= 5
    ax.axhline(E, color='gray', lw=0.6, ls=':')
    ax.plot(x[m], E + 0.9*u[m], lw=1.6,
            label=f'n={n}: E={E:.6f} (exact: {n+0.5})')
    # classical turning points: V(x)=E -> x = sqrt(2E)
    xt = np.sqrt(2*E)
    ax.plot([-xt, xt], [E, E], 'r|', ms=14, mew=2)
ax.set_ylim(0, n_show + 1.2); ax.set_xlim(-5, 5)
ax.set_xlabel('x (oscillator units)')
ax.set_ylabel('energy  /  E + ψ(x) offset')
ax.set_title('Wavefunctions drawn at their own levels; red bars = classical turning points')
ax.legend(fontsize=8, loc='upper center')
plt.show()

The horizontal axis is position and the vertical one is energy; each
wavefunction is shifted up to its level $E_n$, as usual. The red bars mark the
classical turning points ($V = E$), and the tail of the wavefunction reaches
past them: classically forbidden, quantum mechanically accessible, and it is
the same mathematics as the tunnelling of §4. The node count comes out exactly
$n$.

Two variations worth trying: with `n_show = 8` the spacing is still 1; and
replacing `V_oscillator` by `0.5*x**2 + 0.1*x**4` destroys the equal spacing,
which is the signature of anharmonicity.

---
## 2. Hydrogen atom

$$V(r) = -\frac{1}{r}$$

In [ ]:
levels = sq.eigenvalues(sq.V_hydrogen(0), 1e-6, 60, -0.6, -0.03,
                        n_states=3, dx=2e-3)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
for E, nodes in levels:
    n = nodes + 1
    ax1.axhline(E, lw=2)
    ax1.annotate(f'  n={n}: E={E:.5f}  (exact −1/2n² = {-0.5/n**2:.5f})',
                 (0.02, E), fontsize=9)
ax1.axhline(0, color='k', lw=0.8)
ax1.annotate('E=0: ionisation — above it, the continuum\n(which is where SCATTERING lives)',
             (0.02, 0.012), fontsize=9, color='tab:red')
ax1.set_ylim(-0.56, 0.06); ax1.set_xticks([])
ax1.set_ylabel('energy (atomic units)')
ax1.set_title('Levels ACCUMULATING at E=0 (the Rydberg series)')
# 1s: numerical vs analytic
E1 = levels[0][0]
r, u = sq.eigenfunction(sq.V_hydrogen(0), E1, 1e-6, 60, dx=2e-3)
m = r <= 8
ax2.plot(r[m], u[m], 'r-', lw=2, label='u₁ₛ numerical')
ax2.plot(r[m], 2*r[m]*np.exp(-r[m]), 'k--', lw=1.2,
         label='analytic: u = 2r·e⁻ʳ')
ax2.set_xlabel('r (Bohr radii)'); ax2.set_ylabel('u(r)')
ax2.set_title('Ground state: numerical sitting on top of exact')
ax2.legend()
plt.tight_layout(); plt.show()

On the left, the levels $-0.5$, $-0.125$ and $-0.056$ a.u. pile up against
$E = 0$: infinitely many states fit because $-1/r$ is long ranged, unlike the
short-range potentials of this laboratory, which hold zero or one. Above zero
lies the continuum, which is the domain of scattering. On the right, the
numerical $u_{1s}$ lands on $2re^{-r}$.

---
## 3. Morse potential

$$V(r) = D\left(1 - e^{-a(r-r_e)}\right)^2 - D
\qquad\Rightarrow\qquad
E_n = -D + \omega\left(n+\tfrac12\right) - \frac{\omega^2 (n+\tfrac12)^2}{4D},\quad \omega = a\sqrt{2D}$$

The quadratic term is the anharmonicity: levels crowd together as they rise,
and the well holds a finite number of them, $n \le \sqrt{2D}/a - \tfrac12$.
That is the contrast with the oscillator, which holds infinitely many, and it
is the behaviour of a realistic molecular potential such as Lennard-Jones.

In [ ]:
D, a_m, re = 10.0, 1.0, 2.0
V = sq.V_morse(D, a_m, re)
levels = sq.eigenvalues(V, 0.05, 12, -9.99, -0.02, n_states=4, dx=4e-3)
rr = np.linspace(0.4, 9, 400)
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(rr, V(rr), 'k-', lw=2, label='Morse (D=10, a=1)')
w = a_m*np.sqrt(2*D)
for E, n in levels:
    ax.axhline(E, color='tab:purple', lw=1.5,
               xmin=0.05, xmax=0.6)
    E_h = -D + w*(n+0.5)           # equivalent oscillator, no anharmonicity
    ax.axhline(E_h, color='tab:orange', lw=1, ls='--', xmin=0.62, xmax=0.95)
    ax.annotate(f'n={n}', (0.5, E), fontsize=9, color='tab:purple')
ax.annotate('purple: Morse, numerical\n(levels crowding)', (5.5, -8.6),
            fontsize=9, color='tab:purple')
ax.annotate('orange: equivalent oscillator\n(fixed spacing)', (5.5, -2.0),
            fontsize=9, color='tab:orange')
ax.axhline(0, color='k', lw=0.6)
ax.set_ylim(-10.5, 2); ax.set_xlabel('r (interatomic distance)')
ax.set_ylabel('energy'); ax.set_title(
    f'Anharmonicity: only {sq.n_max_morse(D, a_m)+1} levels fit; above E=0 the molecule dissociates')
ax.legend(loc='lower right')
plt.show()
for E, n in levels:
    print(f'n={n}: E_num={E:.5f}  E_exact={sq.E_morse(n, D, a_m):.5f}')

The numerical levels crowd together as they rise, because the well widens
near dissociation. The contrast with an oscillator of the same bottom, whose
spacing is constant, is precisely what a molecular vibrational spectrum
measures. Above $E = 0$ the molecule dissociates, and again there is the
continuum.

---
## 4. Tunnelling

Square barrier of height $V_0$ and width $L$, with $E < V_0$:

$$T(E) = \left[1 + \frac{V_0^2 \sinh^2(\kappa L)}{4E(V_0 - E)}\right]^{-1},
\qquad \kappa = \sqrt{2(V_0 - E)}$$

$T$ is the probability of getting through. Classically $T = 0$ below the top
and $T = 1$ above it. Quantum mechanically the value is exponentially small
but not zero below the barrier, and stays under 1 just above it, with
oscillations.

In [ ]:
V0, L = 5.0, 1.0
Es = np.linspace(0.2, 14, 120)
T_num = [sq.transmission(sq.V_barrier(V0, L), E, -0.5, 1.5, dx=1e-4)
         for E in Es]
T_exact = [sq.T_barrier_exact(E, V0, L) for E in Es]
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.semilogy(Es, T_num, 'r-', lw=2, label='numerical (transfer matrix)')
ax.semilogy(Es, T_exact, 'k--', lw=1.2, label='closed form')
ax.axvline(V0, color='gray', ls=':')
ax.annotate('E = V₀: top of the barrier', (V0, 2e-3), rotation=90, fontsize=9)
ax.annotate('classically FORBIDDEN:\nT ~ e^(−2κL), tiny but ≠ 0',
            (1.2, 3e-3), fontsize=9)
ax.annotate('above the top: T < 1\nwith oscillations (interference\nat the barrier edges)',
            (8.5, 0.35), fontsize=9)
ax.set_xlabel('energy E  (V₀ = 5)')
ax.set_ylabel('transmission T  (LOG scale)')
ax.set_title('Tunnelling: numerical on top of exact across 5 decades')
ax.legend(loc='lower right')
plt.show()

The vertical axis is logarithmic and the curve sweeps five orders of
magnitude. Below $V_0$ the decay is exponential, and it is the same
$e^{-\kappa x}$ as the tail of the oscillator and of a shallow bound state.
Above, $T$ oscillates and only tends to 1: the edges of the barrier reflect,
in the same way that glass interfaces reflect light. The maxima with $T = 1$
are transmission resonances.

Two variations worth trying: doubling `L` makes the tunnelling collapse, since
the dependence on width is exponential; and turning the barrier into a well,
with `V0 = -5`, produces resonances above the well.

---
## Summary

| System | What it establishes |
|---|---|
| Oscillator | quantisation, zero-point energy, forbidden region |
| Hydrogen | the same radial $u(r)$; continuum at $E > 0$ |
| Morse | anharmonicity, finitely many levels, dissociation |
| Tunnelling | $e^{-\kappa x}$ crossing barriers |

The common thread is that **bound states ($E < 0$) and scattering ($E > 0$)
are the two halves of the same equation**, solved by the same integrator.